In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import asyncio
try:
    from ib_async import IB, Stock
except ImportError:
    from ib_insync import IB, Stock

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Market Data" / "Cleaned"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PARQUET = DATA_DIR / "etf_prices_master.parquet"
MASTER_CSV = DATA_DIR / "etf_prices_master.csv"

HOST = "127.0.0.1"
PORT = 4004          # paper
CLIENT_ID = 106

ACTIVE_UNIVERSE = [
    "SPY", "QQQ",
    "VPU", "VFH", "VHT", "VIS",
    "VWO", "INDA", "VEA",
    "SHY", "TLT", "BIL", "IEI",
    "UNG", "SLV", "GLD", "VNQ", "USO"
]

In [3]:
master = pd.read_parquet(MASTER_PARQUET)
master["date"] = pd.to_datetime(master["date"]).dt.normalize()

print(master.shape)
display(master.tail())

(103131, 9)


,date,symbol,open,high,low,close,volume,nav,total_return_gross
103126,2026-03-02,VWO,56.81,57.50,56.76,57.24,12725480.0,57.19,100.12
103127,2026-03-03,VWO,54.88,55.48,53.94,55.23,31763840.0,55.33,96.60
103128,2026-03-04,VWO,55.42,55.67,55.09,55.58,21718700.0,55.49,97.22
103129,2026-03-05,VWO,54.94,55.35,54.24,54.85,15624100.0,54.96,95.94
103130,2026-03-06,VWO,54.33,54.78,54.12,54.47,9830010.0,NaN,95.27


In [4]:
ib = IB()
await ib.connectAsync(HOST, PORT, clientId=CLIENT_ID)
print("Connected:", ib.isConnected())

Connected: True


In [5]:
def fetch_daily_bars(symbol: str, lookback: str = "15 D") -> pd.DataFrame:
    contract = Stock(symbol, "SMART", "USD")
    ib.qualifyContracts(contract)

    bars = ib.reqHistoricalData(
        contract,
        endDateTime="",
        durationStr=lookback,
        barSizeSetting="1 day",
        whatToShow="TRADES",
        useRTH=True,
        formatDate=1,
    )

    rows = []
    for bar in bars:
        rows.append({
            "date": pd.to_datetime(bar.date).normalize(),
            "symbol": symbol,
            "open": float(bar.open),
            "high": float(bar.high),
            "low": float(bar.low),
            "close": float(bar.close),
            "volume": float(bar.volume) if bar.volume is not None else np.nan,
            "nav": np.nan,
            "total_return_gross": np.nan,
        })

    return pd.DataFrame(rows)

In [6]:
ibkr_frames = []

for symbol in ACTIVE_UNIVERSE:
    try:
        df_symbol = fetch_daily_bars(symbol)
        if not df_symbol.empty:
            ibkr_frames.append(df_symbol)
            print(f"{symbol}: {len(df_symbol)} rows")
        else:
            print(f"{symbol}: no rows returned")
    except Exception as e:
        print(f"{symbol}: ERROR -> {e}")

ibkr_df = pd.concat(ibkr_frames, ignore_index=True) if ibkr_frames else pd.DataFrame()

print(ibkr_df.shape)
display(ibkr_df.tail(20))

SPY: ERROR -> This event loop is already running
QQQ: ERROR -> This event loop is already running
VPU: ERROR -> This event loop is already running
VFH: ERROR -> This event loop is already running
VHT: ERROR -> This event loop is already running
VIS: ERROR -> This event loop is already running
VWO: ERROR -> This event loop is already running
INDA: ERROR -> This event loop is already running
VEA: ERROR -> This event loop is already running
SHY: ERROR -> This event loop is already running
TLT: ERROR -> This event loop is already running
BIL: ERROR -> This event loop is already running
IEI: ERROR -> This event loop is already running
UNG: ERROR -> This event loop is already running
SLV: ERROR -> This event loop is already running
GLD: ERROR -> This event loop is already running
VNQ: ERROR -> This event loop is already running
USO: ERROR -> This event loop is already running
(0, 0)


""


In [7]:
ib.disconnect()
print("Disconnected")

Disconnected


In [8]:
expected_cols = [
    "date", "symbol", "open", "high", "low", "close",
    "volume", "nav", "total_return_gross"
]

for col in expected_cols:
    if col not in master.columns:
        master[col] = np.nan
    if col not in ibkr_df.columns:
        ibkr_df[col] = np.nan

master = master[expected_cols].copy()
ibkr_df = ibkr_df[expected_cols].copy()

combined = pd.concat([master, ibkr_df], ignore_index=True)
combined["date"] = pd.to_datetime(combined["date"]).dt.normalize()

combined = combined.drop_duplicates(subset=["symbol", "date"], keep="last")
combined = combined.sort_values(["symbol", "date"]).reset_index(drop=True)

print("Old master shape:", master.shape)
print("New IBKR rows:", ibkr_df.shape)
print("Combined shape:", combined.shape)

Old master shape: (103131, 9)
New IBKR rows: (0, 9)
Combined shape: (103131, 9)


/var/folders/wk/189ck3455t5d1pp0p1ydbpxw0000gp/T/ipykernel_52361/1666031919.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([master, ibkr_df], ignore_index=True)


In [9]:
latest_rows = combined.groupby("symbol", as_index=False).tail(3)
display(latest_rows.sort_values(["symbol", "date"]))

,date,symbol,open,high,low,close,volume,nav,total_return_gross
4718,2026-03-04,BIL,91.41,91.41,91.40,91.41,9932489.0,91.40,117.61
4719,2026-03-05,BIL,91.42,91.42,91.41,91.42,9248953.0,91.41,117.62
4720,2026-03-06,BIL,91.45,91.45,91.44,91.44,8733211.0,NaN,117.65
5124,2026-03-04,ETHA,15.71,16.63,15.60,16.25,69898805.0,16.39,16.25
5125,2026-03-05,ETHA,16.00,16.11,15.50,15.80,45188900.0,15.76,15.80
...,...,...,...,...,...,...,...,...,...
97849,2026-03-05,VPU,202.86,202.86,200.71,202.48,319224.0,202.45,416.63
97850,2026-03-06,VPU,201.22,202.54,199.98,201.60,224477.0,NaN,414.82
103128,2026-03-04,VWO,55.42,55.67,55.09,55.58,21718700.0,55.49,97.22
103129,2026-03-05,VWO,54.94,55.35,54.24,54.85,15624100.0,54.96,95.94


In [10]:
combined.to_parquet(MASTER_PARQUET, index=False)
combined.to_csv(MASTER_CSV, index=False)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
append_snapshot_csv = DATA_DIR / f"ibkr_daily_append_{timestamp}.csv"
ibkr_df.to_csv(append_snapshot_csv, index=False)

print("Saved:")
print(MASTER_PARQUET)
print(MASTER_CSV)
print(append_snapshot_csv)

Saved:
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_prices_master.parquet
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_prices_master.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/ibkr_daily_append_20260309_104837.csv


In [11]:
check = combined.duplicated(subset=["symbol", "date"]).sum()
print("Duplicate symbol-date rows:", check)
print("Date range:", combined["date"].min(), "to", combined["date"].max())
print("Symbols:", combined["symbol"].nunique())

Duplicate symbol-date rows: 0
Date range: 1993-02-01 00:00:00 to 2026-03-06 00:00:00
Symbols: 22
